# ICT-15i -- Pont #1-bis : le paysage 2D anisotrope clot l'Epic

Les chantiers 1/3 (`ICT-15f`, double-puits symetrique 1D) et 2/3
(`ICT-15h`, double-puits asymetrique 1D) ont tranche **CONFIRMED-NEGATIVE** : la
courbure locale $\sigma$ n'a aucun pouvoir predictif propre pour la
recuperabilite une fois la largeur ET la barriere controlees.

Mais le regime 1D a une limite de principe : la courbure locale est un scalaire.
En 2D, le Hessien au minimum est une matrice -- ses **valeurs propres** donnent une
courbure par direction principale. Un resume scalaire ($\sigma_{\text{mean}}$ ou
$\sigma_{\text{min}}$) peut cacher la geometrie. Ce chantier 3/3 teste la
**generalisation 2D**, dans le regime le plus favorable a l'hypothese
*« $\sigma$ cause la recuperabilite »* : le paysage **anisotrope** ou l'on peut
varier la courbure transverse **sans toucher** a la largeur du puits.

**La nouveaute methodologique (vs 2/3).** En regime asymetrique 1D, $\sigma$ et la
largeur etaient structurellement couples intra-puits (impossible a separer). En
2D, l'axe transverse orthogonal ($y$) offre un **knob genuinely 2D** : on atteint
le decouplage par construction ($|\rho|<0.2$) que le chantier 2/3 ne pouvait pas.
Et malgre cela -- avec le scalaire le plus charitable ($\sigma_{\text{min}}$, la
direction molle de l'escape) -- le verdict reste negatif. L'Epic se clot.

> **Statut épistémique** — **Sans verdict à ce jour** : aucune ligne de la [matrice de dissociations](../../../../docs/ict/dissociations-matrix.md) ne concerne ce notebook ; son statut épistémique sera porté par la matrice le cas échéant.

## 1. Le substrat : double-puits 2D anisotrope $V(x,y) = a x^4 - b x^2 + d y^2$

L'axe $x$ porte le double-puits (2 minima en $\pm\sqrt{b/2a}$, 1 selle en
l'origine). L'axe $y$ est un **confinement harmonique** de raideur $d$. Le Hessien
au minimum est diagonal :

$$H(x^*) = \begin{pmatrix} 4b & 0 \\ 0 & 2d \end{pmatrix} \quad\Rightarrow\quad \lambda_x = 4b,\ \lambda_y = 2d.$$

La courbure transverse $\lambda_y = 2d$ est un **knob orthogonal** a la largeur
longitudinale $w_x = \sqrt{b/2a}$ : on peut la faire varier sans toucher au profil
du double-puits. C'est l'avantage genuinely 2D, impossible en 1D ou courbure et
largeur partagent les memes parametres.

In [1]:
using System;
using System.Linq;
using System.Text;
using System.Globalization;
using Microsoft.DotNet.Interactive.Formatting;

CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.CurrentUICulture = CultureInfo.InvariantCulture;

// Paysage canonique (a,b,d) = (1, 2, 3) : minima (+-1, 0), selle (0,0).
double a = 1.0, b = 2.0, d = 3.0;
Func<double, double, double> potentiel = (x, y) => a * Math.Pow(x, 4) - b * x * x + d * y * y;
var eqs = new[] { (x: -Math.Sqrt(b / (2 * a)), y: 0.0, type: "minimum"), (x: 0.0, y: 0.0, type: "saddle"), (x: Math.Sqrt(b / (2 * a)), y: 0.0, type: "minimum") };

// Grille pour la visualisation (le potentiel vectorise sur la grille directement).
int nx = 220, ny = 160;
double[] xs = Enumerable.Range(0, nx).Select(i => -1.8 + 3.6 * i / (nx - 1)).ToArray();
double[] ys = Enumerable.Range(0, ny).Select(j => -1.2 + 2.4 * j / (ny - 1)).ToArray();
double[,] z = new double[nx, ny];
for (int i = 0; i < nx; i++)
    for (int j = 0; j < ny; j++) z[i, j] = potentiel(xs[i], ys[j]);
double zmin = z.Cast<double>().Min(), zmax = z.Cast<double>().Max();
Func<double, double> sx = x => 65 + (x + 1.8) / 3.6 * 510;
Func<double, double> sy = y => 340 - (y + 1.2) / 2.4 * 280;
Func<double, string> fmt = v => v.ToString("F3", CultureInfo.InvariantCulture);
var svg = new StringBuilder();
svg.Append("<svg xmlns='http://www.w3.org/2000/svg' width='700' height='420' viewBox='0 0 700 420'><rect width='700' height='420' fill='white'/>");
// Palette approchant viridis et niveaux remplis sur la grille d'origine.
Func<double, string> couleur = t => { t = Math.Max(0, Math.Min(1, t)); int r = (int)(68 + 185 * t * t), g = (int)(1 + 230 * t), bl = (int)(84 + 55 * Math.Sin(Math.PI * t) - 48 * t); return $"rgb({r},{g},{bl})"; };
for (int i = 0; i < nx - 1; i++)
    for (int j = 0; j < ny - 1; j++)
    {
        double t = (z[i, j] - zmin) / (zmax - zmin);
        t = Math.Floor(t * 30) / 29.0;
        svg.Append($"<rect x='{fmt(sx(xs[i]))}' y='{fmt(sy(ys[j + 1]))}' width='{fmt(510.0 / (nx - 1) + 0.1)}' height='{fmt(280.0 / (ny - 1) + 0.1)}' fill='{couleur(t)}'/>");
    }
// Courbes de niveau par interpolation linéaire sur les arêtes de chaque case.
for (int k = 1; k <= 12; k++)
{
    double level = zmin + (zmax - zmin) * k / 13.0;
    for (int i = 0; i < nx - 1; i++)
        for (int j = 0; j < ny - 1; j++)
        {
            double[] vx = { xs[i], xs[i + 1], xs[i + 1], xs[i] };
            double[] vy = { ys[j], ys[j], ys[j + 1], ys[j + 1] };
            double[] vz = { z[i, j], z[i + 1, j], z[i + 1, j + 1], z[i, j + 1] };
            var points = new System.Collections.Generic.List<(double x, double y)>();
            for (int e = 0; e < 4; e++)
            {
                int q = (e + 1) % 4;
                if ((vz[e] < level) != (vz[q] < level))
                {
                    double t = (level - vz[e]) / (vz[q] - vz[e]);
                    points.Add((vx[e] + t * (vx[q] - vx[e]), vy[e] + t * (vy[q] - vy[e])));
                }
            }
            for (int q = 0; q + 1 < points.Count; q += 2)
                svg.Append($"<path d='M {fmt(sx(points[q].x))} {fmt(sy(points[q].y))} L {fmt(sx(points[q + 1].x))} {fmt(sy(points[q + 1].y))}' stroke='black' stroke-opacity='.5' stroke-width='.4' fill='none'/>");
        }
}
foreach (var eq in eqs)
{
    // plot (pas scatter) pour eviter le warning matplotlib "edgecolor for unfilled
    // marker" dont le traceback injecte le chemin ipykernel dans la sortie.
    double px = sx(eq.x), py = sy(eq.y);
    if (eq.type == "saddle")
        svg.Append($"<path d='M {fmt(px - 6)} {fmt(py - 6)} l 12 12 M {fmt(px - 6)} {fmt(py + 6)} l 12 -12' stroke='red' stroke-width='2.5'/>");
    else
        svg.Append($"<circle cx='{fmt(px)}' cy='{fmt(py)}' r='6' fill='none' stroke='#d62728' stroke-width='2'/>");
}
svg.Append("<rect x='65' y='60' width='510' height='280' fill='none' stroke='black'/><text x='230' y='28' font-size='16'>V(x,y)=a x⁴ - b x² + d y² (a=1, b=2, d=3)</text><text x='260' y='382' font-size='14'>x (axe double-puits)</text><text transform='translate(18 275) rotate(-90)' font-size='14'>y (axe transverse)</text>");
for (int k = 0; k < 100; k++) svg.Append($"<rect x='600' y='{fmt(60 + 2.8 * k)}' width='18' height='2.9' fill='{couleur(1 - k / 99.0)}'/>");
svg.Append($"<text x='625' y='70' font-size='12'>{fmt(zmax)}</text><text x='625' y='340' font-size='12'>{fmt(zmin)}</text><text x='605' y='355' font-size='13'>V</text></svg>");
// Enregistrer explicitement le formateur SVG avant l'affichage.
Formatter.Register<string>((value, writer) => writer.Write(value), "image/svg+xml");
display(FormattedValue.CreateSingleFromObject(svg.ToString(), "image/svg+xml"));
Console.WriteLine("Minima : ['minimum', 'minimum'] | Selle : ['saddle']");

image/svg+xml: <svg xmlns='http://www.w3.org/2000/svg' width='700 ... MimeType image/svg+xml Value <svg xmlns='http://www.w3.org/2000/svg' width='700' height='420' viewBox='0 0 700 420'><rect width='700' height='420' fill='white'/><rect x='65.000' y='338.239' width='2.429' height='1.861' fill='rgb(253,231,36)'/><rect x='65.000' y='336.478' width='2.429' height='1.861' fill='rgb(253,231,36)'/><rect x='65.000' y='334.717' width='2.429' height='1.861' fill='rgb(253,231,36)'/><rect x='65.000' y='332.956' width='2.429' height='1.861' fill='rgb(240,223,43)'/><rect x='65.000' y='331.195' width='2.429' height='1.861' fill='rgb(240,223,43)'/><rect x='65.000' y='329.434' width='2.429' height='1.861' fill='rgb(240,223,43)'/><rect x='65.000' y='327.673' width='2.429' height='1.861' fill='rgb(228,215,51)'/><rect x='65.000' y='325.912' width='2.429' height='1.861' fill='rgb(228,215,51)'/><rect x='65.000' y='324.151' width='2.429' height='1.861' fill='rgb(228,215,51)'/><rect x='65.000' y='322.390' width='2.429' height='1.861' fill='rgb(228,215,51)'/><rect x='65.000' y='320.629' width='2.429' height='1.861' fill='rgb(216,207,58)'/><rect x='65.000' y='318.868' width='2.429' height='1.861' fill='rgb(216,207,58)'/><rect x='65.000' y='317.107' width='2.429' height='1.861' fill='rgb(216,207,58)'/><rect x='65.000' y='315.346' width='2.429' height='1.861' fill='rgb(205,199,65)'/><rect x='65.000' y='313.585' width='2.429' height='1.861' fill='rgb(205,199,65)'/><rect x='65.000' y='311.824' width='2.429' height='1.861' fill='rgb(205,199,65)'/><rect x='65.000' y='310.063' width='2.429' height='1.861' fill='rgb(194,191,72)'/><rect x='65.000' y='308.302' width='2.429' height='1.861' fill='rgb(194,191,72)'/><rect x='65.000' y='306.541' width='2.429' height='1.861' fill='rgb(194,191,72)'/><rect x='65.000' y='304.780' width='2.429' height='1.861' fill='rgb(194,191,72)'/><rect x='65.000' y='303.019' width='2.429' height='1.861' fill='rgb(184,183,79)'/><rect x='65.000' y='301.258' width='2.429' height='1.861' fill='rgb(184,183,79)'/><rect x='65.000' y='299.497' width='2.429' height='1.861' fill='rgb(184,183,79)'/><rect x='65.000' y='297.736' width='2.429' height='1.861' fill='rgb(184,183,79)'/><rect x='65.000' y='295.975' width='2.429' height='1.861' fill='rgb(174,175,85)'/><rect x='65.000' y='294.214' width='2.429' height='1.861' fill='rgb(174,175,85)'/><rect x='65.000' y='292.453' width='2.429' height='1.861' fill='rgb(174,175,85)'/><rect x='65.000' y='290.692' width='2.429' height='1.861' fill='rgb(174,175,85)'/><rect x='65.000' y='288.931' width='2.429' height='1.861' fill='rgb(165,167,91)'/><rect x='65.000' y='287.170' width='2.429' height='1.861' fill='rgb(165,167,91)'/><rect x='65.000' y='285.409' width='2.429' height='1.861' fill='rgb(165,167,91)'/><rect x='65.000' y='283.648' width='2.429' height='1.861' fill='rgb(165,167,91)'/><rect x='65.000' y='281.887' width='2.429' height='1.861' fill='rgb(165,167,91)'/><rect x='65.000' y='280.126' width='2.429' height='1.861' fill='rgb(155,159,96)'/><rect x='65.000' y='278.365' width='2.429' height='1.861' fill='rgb(155,159,96)'/><rect x='65.000' y='276.604' width='2.429' height='1.861' fill='rgb(155,159,96)'/><rect x='65.000' y='274.843' width='2.429' height='1.861' fill='rgb(155,159,96)'/><rect x='65.000' y='273.082' width='2.429' height='1.861' fill='rgb(155,159,96)'/><rect x='65.000' y='271.321' width='2.429' height='1.861' fill='rgb(147,151,101)'/><rect x='65.000' y='269.560' width='2.429' height='1.861' fill='rgb(147,151,101)'/><rect x='65.000' y='267.799' width='2.429' height='1.861' fill='rgb(147,151,101)'/><rect x='65.000' y='266.038' width='2.429' height='1.861' fill='rgb(147,151,101)'/><rect x='65.000' y='264.277' width='2.429' height='1.861' fill='rgb(147,151,101)'/><rect x='65.000' y='262.516' width='2.429' height='1.861' fill='rgb(147,151,101)'/><rect x='65.000' y='260.755' width='2.429' height='1.861' fill='rgb(139,143,105)'/><rect x='65.000' y='258.994' width='2.429' height='1.861' fill='rgb(139,

Minima : ['minimum', 'minimum'] | Selle : ['saddle']


Les trois **valeurs propres** attendues au minimum sont $\lambda_x = 4b = 8$ et
$\lambda_y = 2d = 6$ (Hessien diagonal). Verifions que le profilage
substrate-agnostic de `basin_geometry` les recupere, ainsi que la forme fermee de
la largeur et de la barriere.

In [2]:
// Coherence geometrique : formes fermees vs profilage numerique.
// Dérivées centrales et recherche de Newton sur la même grille que le profileur Python.
Func<Func<double, double, double>, double, double, (double gx, double gy)> gradient = (v, x, y) =>
{
    double h = 1e-5;
    return ((v(x + h, y) - v(x - h, y)) / (2 * h), (v(x, y + h) - v(x, y - h)) / (2 * h));
};
Func<Func<double, double, double>, double, double, (double xx, double xy, double yy)> hessian = (v, x, y) =>
{
    double h = 1e-4, f = v(x, y);
    return ((v(x + h, y) - 2 * f + v(x - h, y)) / (h * h),
            (v(x + h, y + h) - v(x + h, y - h) - v(x - h, y + h) + v(x - h, y - h)) / (4 * h * h),
            (v(x, y + h) - 2 * f + v(x, y - h)) / (h * h));
};
var roots = new System.Collections.Generic.List<(double x, double y, double lo, double hi)>();
for (int i = 0; i < 30; i++)
    for (int j = 0; j < 30; j++)
    {
        double x = -2.0 + 4.0 * i / 29.0, y = -2.0 + 4.0 * j / 29.0;
        bool converged = false;
        for (int iter = 0; iter < 50; iter++)
        {
            var g = gradient(potentiel, x, y);
            if (Math.Sqrt(g.gx * g.gx + g.gy * g.gy) < 1e-9) { converged = true; break; }
            var h = hessian(potentiel, x, y);
            double det = h.xx * h.yy - h.xy * h.xy;
            if (Math.Abs(det) < 1e-30) break;
            double dx = (h.yy * g.gx - h.xy * g.gy) / det;
            double dy = (h.xx * g.gy - h.xy * g.gx) / det;
            x -= dx; y -= dy;
            if (double.IsNaN(x) || double.IsInfinity(x) || double.IsNaN(y) || double.IsInfinity(y)) break;
        }
        if (!converged || roots.Any(r => Math.Sqrt((r.x - x) * (r.x - x) + (r.y - y) * (r.y - y)) < 1e-3)) continue;
        var hh = hessian(potentiel, x, y);
        double disc = Math.Sqrt((hh.xx - hh.yy) * (hh.xx - hh.yy) + 4 * hh.xy * hh.xy);
        roots.Add((x, y, (hh.xx + hh.yy - disc) / 2, (hh.xx + hh.yy + disc) / 2));
    }
var p = roots.First(r => r.lo > 0);
var col = roots.Where(r => r.lo <= 0 && r.hi >= 0).OrderBy(r => (r.x - p.x) * (r.x - p.x) + (r.y - p.y) * (r.y - p.y)).First();
double width = Math.Sqrt((p.x - col.x) * (p.x - col.x) + (p.y - col.y) * (p.y - col.y));
double barrier = potentiel(col.x, col.y) - potentiel(p.x, p.y);
Console.WriteLine(FormattableString.Invariant($"lambda_x (attendu 4b={4*b:F2}) : numerique {p.hi:F3}"));
Console.WriteLine(FormattableString.Invariant($"lambda_y (attendu 2d={2*d:F2}) : numerique {p.lo:F3}"));
Console.WriteLine(FormattableString.Invariant($"anisotropie lambda_max/lambda_min : {p.hi / p.lo:F3}"));
Console.WriteLine(FormattableString.Invariant($"width (attendu sqrt(b/2a)={Math.Sqrt(b / (2*a)):F3}) : {width:F3}"));
Console.WriteLine(FormattableString.Invariant($"barrier (attendu b^2/4a={b*b/(4*a):F2}) : {barrier:F3}"));

lambda_x (attendu 4b=8.00) : numerique 8.000


lambda_y (attendu 2d=6.00) : numerique 6.000


anisotropie lambda_max/lambda_min : 1.333


width (attendu sqrt(b/2a)=1.000) : 1.000


barrier (attendu b^2/4a=1.00) : 1.000


## Lecture 1 -- Formes fermees vs profilage : 8.000, 6.000, 1.333

La coherence geometrique tient : lambda_x numerique = **8.000** pour une forme fermee 4b = 8.00, lambda_y = **6.000** pour 2d = 6.00 -- le Hessien diagonal du paysage canonique (a, b, d) = (1, 2, 3) est bien recupere par le profilage substrate-agnostic de `basin_geometry`. L'anisotropie lambda_max/lambda_min = 1.333, la largeur = 1.000 (forme fermee sqrt(b/2a) = 1.000) et la barriere = 1.000 (forme fermee b^2/4a = 1.00) : tout coincide au troisieme chiffre.

Le point methodologique est la distinction des deux valeurs propres (8 vs 6) des le canonique : le Hessien 2D au minimum est une matrice, pas un scalaire -- c'est le fondement du chantier 3/3, une courbure par direction principale, et le pretexte de la question des resumes scalaires (sigma_min vs sigma_mean) tranchee a la cellule 12.

## 2. Le knob transverse : $d$ varie la courbure SANS toucher la largeur

**C'est le fait cle genuinely 2D.** En 1D, courbure $\sigma$ et largeur $w$
partagent les memes parametres ($\sigma \propto b$, $w \propto \sqrt{b}$) : on ne
peut pas faire varier l'une sans l'autre. En 2D, fixer $(a,b)$ fige le profil
longitudinal (largeur ET barriere du double-puits), et l'axe $y$ ajoute une
courbure transverse $\lambda_y = 2d$ **independante**. Varier $d$ fait varier
$\lambda_y$ et l'anisotropie, tout en laissant $w_x$ invariant.

In [3]:
// LE KNOB TRANSVERSE : a,b fixes -> width fixe ; varier d -> lambda_y et anisotropie.
a = 1.0; b = 2.0;
var ds = new[] { 0.5, 1.5, 3.0, 6.0 };
var widths = new System.Collections.Generic.List<double>();
var lambdaYs = new System.Collections.Generic.List<double>();
var anisos = new System.Collections.Generic.List<double>();
foreach (double dval in ds)
{
    Func<double, double, double> v = (x, y) => a * Math.Pow(x, 4) - b * x * x + dval * y * y;
    var points = new System.Collections.Generic.List<(double x, double y, double lo, double hi)>();
    for (int i = 0; i < 30; i++)
        for (int j = 0; j < 30; j++)
        {
            double x = -2.0 + 4.0 * i / 29.0, y = -2.0 + 4.0 * j / 29.0;
            bool converged = false;
            for (int iteration = 0; iteration < 50; iteration++)
            {
                var g = gradient(v, x, y);
                if (Math.Sqrt(g.gx * g.gx + g.gy * g.gy) < 1e-9) { converged = true; break; }
                var h = hessian(v, x, y);
                double det = h.xx * h.yy - h.xy * h.xy;
                if (Math.Abs(det) < 1e-30) break;
                double dx = (h.yy * g.gx - h.xy * g.gy) / det;
                double dy = (h.xx * g.gy - h.xy * g.gx) / det;
                x -= dx; y -= dy;
                if (double.IsNaN(x) || double.IsInfinity(x) || double.IsNaN(y) || double.IsInfinity(y)) break;
            }
            if (!converged || points.Any(q => Math.Sqrt((q.x-x)*(q.x-x)+(q.y-y)*(q.y-y)) < 1e-3)) continue;
            var hh = hessian(v, x, y);
            double disc = Math.Sqrt((hh.xx-hh.yy)*(hh.xx-hh.yy)+4*hh.xy*hh.xy);
            points.Add((x, y, (hh.xx+hh.yy-disc)/2, (hh.xx+hh.yy+disc)/2));
        }
    var min = points.First(q => q.lo > 0);
    var saddle = points.Where(q => q.lo <= 0 && q.hi >= 0)
        .OrderBy(q => (q.x-min.x)*(q.x-min.x)+(q.y-min.y)*(q.y-min.y)).First();
    widths.Add(Math.Sqrt((min.x-saddle.x)*(min.x-saddle.x)+(min.y-saddle.y)*(min.y-saddle.y)));
    lambdaYs.Add(min.lo); // lambda_y = 2d (le min)
    anisos.Add(min.hi / min.lo);
}
Console.WriteLine("d     | width_x (invariant) | lambda_y=2d | anisotropie");
for (int i = 0; i < ds.Length; i++)
    Console.WriteLine(FormattableString.Invariant($"{ds[i],-5:F1} | {widths[i]:F4}             | {lambdaYs[i]:F3}      | {anisos[i]:F3}"));
Console.WriteLine(FormattableString.Invariant($"\nwidth invariant (ecart max {widths.Max()-widths.Min():E2}) ; lambda_y varie de {lambdaYs.Min():F1} a {lambdaYs.Max():F1} -> decouplage genuinely 2D."));

d     | width_x (invariant) | lambda_y=2d | anisotropie


0.5   | 1.0000             | 1.000      | 8.000


1.5   | 1.0000             | 3.000      | 2.667


3.0   | 1.0000             | 6.000      | 1.333


6.0   | 1.0000             | 8.000      | 1.500



width invariant (ecart max 4.68E-013) ; lambda_y varie de 1.0 a 8.0 -> decouplage genuinely 2D.


## Lecture 2 -- Le knob transverse : width 1.0000 invariant, lambda_y 1.0 -> 8.0

Le tableau des quatre valeurs de d est le fait cle du chantier : d = 0.5, 1.5, 3.0, 6.0 donne les valeurs propres minimales 1.000, 3.000, 6.000, 8.000 -- pendant que width_x reste **1.0000 partout**, avec un ecart maximum de 4.68e-13 (invariant au bruit numerique pres). L'anisotropie vaut 8.000, 2.667, 1.333, 1.500 : elle diminue puis augmente lorsque la direction transverse devient la plus raide.

La courbure transverse $\lambda_y=2d$ vaut quant à elle 1, 3, 6 et 12 : pour d = 6, la valeur propre minimale affichée est la courbure longitudinale 8, et non $\lambda_y$. Fixer (a, b) fige la largeur ET la barriere du double-puits, tandis que l'axe y ajoute une courbure transverse independante. Les conditions du verdict de la cellule 12 sont en place : varier la courbure sans confondre avec la geometrie globale.

## 3. La mesure NON triviale : recuperation stochastique 2D (Langevin)

Comme en 1D, la recuperation est mesuree par une dynamique de Langevin perturbee
depuis un minimum, puis un test de bassin : la trajectoire retombe-t-elle dans le
memes bassin ($\operatorname{sign}(x_f)=\operatorname{sign}(x^*)$) ?

Deux precautions 2D :
1. **Variance reelle** : sur la famille 2D, la fraction recuperee n'est pas
   constante -- la mesure est falsifiable.
2. **Kick transverse** : sans exciter la direction $y$, la dynamique reste sur
   l'axe $x$ et la direction $d$ n'est jamais echantillonnee. Un kick transverse
   initial est necessaire pour que la recuperation depende effectivement de $d$.

In [4]:
// Générateur xorshift64* et normales de Box-Muller, à graine explicite.
public class LandscapeRandom
{
    private ulong state;
    private bool hasSpare;
    private double spare;
    public LandscapeRandom(ulong seed) { state = seed == 0 ? 0x9E3779B97F4A7C15UL : seed; }
    public ulong NextUInt64()
    {
        ulong x = state;
        x ^= x >> 12; x ^= x << 25; x ^= x >> 27;
        state = x;
        return unchecked(x * 2685821657736338717UL);
    }
    public double Uniform() => ((NextUInt64() >> 11) + 0.5) / 9007199254740992.0;
    public double Normal()
    {
        if (hasSpare) { hasSpare = false; return spare; }
        double radius = Math.Sqrt(-2.0 * Math.Log(Uniform()));
        double angle = 2.0 * Math.PI * Uniform();
        spare = radius * Math.Sin(angle); hasSpare = true;
        return radius * Math.Cos(angle);
    }
}
// Euler-Maruyama : deux normales par essai et par pas, dans l'ordre du tableau source.
double Recover2D(double xstar, double aa, double bb, double dd, LandscapeRandom random,
    double noise = 0.35, int trials = 60, int steps = 3000, double dt = 0.02,
    double perturb = 0.75, double kick = 0.6)
{
    var xx = Enumerable.Repeat(xstar * perturb, trials).ToArray();
    var yy = Enumerable.Repeat(kick, trials).ToArray();
    double scale = Math.Sqrt(2 * noise * dt), xmax = 10 * (Math.Abs(xstar) + 1);
    for (int t = 0; t < steps; t++)
        for (int i = 0; i < trials; i++)
        {
            double x = xx[i], y = yy[i];
            double nx = random.Normal(), ny = random.Normal();
            xx[i] = Math.Max(-xmax, Math.Min(xmax, x + (2 * bb * x - 4 * aa * x * x * x) * dt + scale * nx));
            yy[i] = y - 2 * dd * y * dt + scale * ny;
        }
    int sign = xstar > 0 ? 1 : -1;
    return (double)xx.Count(x => Math.Sign(x) == sign) / trials;
}

// (a) Variance reelle a travers la famille 2D -> mesure falsifiable.
var rng0 = new LandscapeRandom(0);
var fracs = new System.Collections.Generic.List<double>();
foreach (double aa in new[] { 0.6, 1.0, 2.0 })
    foreach (double bb in new[] { 1.0, 2.0, 3.0 })
        foreach (double dd in new[] { 0.5, 3.0, 6.0 })
        {
            double star = Math.Sqrt(bb / (2 * aa));
            foreach (double xstar in new[] { -star, 0.0, star })
                fracs.Add(Recover2D(xstar, aa, bb, dd, rng0, noise: 0.35, trials: 30, steps: 1500));
        }
double fracMean = fracs.Average();
double fracStd = Math.Sqrt(fracs.Average(f => (f - fracMean) * (f - fracMean)));
Console.WriteLine(FormattableString.Invariant($"Fraction recuperee sur la famille 2D : std={fracStd:F3} (min {fracs.Min():F2}, max {fracs.Max():F2}) -> NON triviale."));

// (b) Sensibilite au kick transverse : sans kick, la direction d n'est pas excitee.
var rng42 = new LandscapeRandom(42);
double fNoKick = Recover2D(1.0, 1.0, 2.0, 0.8, rng42, trials: 60, steps: 2000, kick: 0.0);
double fKick = Recover2D(1.0, 1.0, 2.0, 0.8, rng42, trials: 60, steps: 2000, kick: 0.8);
Console.WriteLine(FormattableString.Invariant($"Recuperation sans kick : {fNoKick:F3} | avec kick : {fKick:F3}"));

// (c) SANITE physique : barriere plus haute -> recuperation plus haute (Arrhenius).
var rngPhysical = new LandscapeRandom(42);
double fLo = Recover2D(0.6, 0.6, 2.0, 2.0, rngPhysical, noise: 0.35, trials: 80, steps: 2000);
double fHi = Recover2D(2.5, 3.5, 2.0, 2.0, rngPhysical, noise: 0.35, trials: 80, steps: 2000);
Console.WriteLine(FormattableString.Invariant($"Barriere faible -> {fLo:F3} | haute -> {fHi:F3} (Arrhenius respecte : {fHi > fLo})."));
Console.WriteLine(FormattableString.Invariant($"Ecarts C# - Python : std={fracStd-0.203:+0.000;-0.000;0.000}, sans kick={fNoKick-0.467:+0.000;-0.000;0.000}, avec kick={fKick-0.483:+0.000;-0.000;0.000}, controle 1={fLo-0.787:+0.000;-0.000;0.000}, controle 2={fHi-0.475:+0.000;-0.000;0.000}."));

Fraction recuperee sur la famille 2D : std=0.187 (min 0.27, max 1.00) -> NON triviale.


Recuperation sans kick : 0.533 | avec kick : 0.400


Barriere faible -> 0.713 | haute -> 0.512 (Arrhenius respecte : False).


Ecarts C# - Python : std=-0.016, sans kick=+0.066, avec kick=-0.083, controle 1=-0.075, controle 2=+0.038.


## Lecture 3 -- Falsifiable, kick transverse et contrôle physique

La mesure est **non triviale** : sur la famille échantillonnée, l'écart-type des fractions récupérées est de 0.187, avec un minimum de 0.27 et un maximum de 1.00. La source Python donne un écart-type de 0.203 : les générateurs aléatoires diffèrent malgré la graine déclarée.

La récupération vaut 0.533 sans kick et 0.400 avec kick. Le bruit agit aussi sur l'axe transverse sans kick ; de plus, dans ce potentiel séparable, la dynamique de $x$ ne dépend pas de $y$. Cette comparaison de deux tirages ne démontre donc pas un effet causal du kick sur la récupération.

Enfin, les deux contrôles affichent 0.713 et 0.512 ; la comparaison annoncée « Arrhenius respecte » est `False`. Ces valeurs diffèrent de celles de la source, mais le résultat booléen est le même.

> **Exercice 1** (barriere ET courbure transverse). La recuperation depend a la
> fois de la barriere longitudinale ($\propto b^2/4a$) et de la courbure
> transverse ($\lambda_y = 2d$). Construire une famille ou ces deux quantites
> varient en sens opposes (barriere croissante, courbure decroissante) et
> afficher la fraction recuperee. Lequel des deux semble piloter la recuperation ?

In [5]:
// Exercice 1 -- a completer.
System.Collections.Generic.List<(double barrier, double lambdaY, double fraction)> RecoveryVsConflictingBarrierCurvature()
{
    var results = new System.Collections.Generic.List<(double barrier, double lambdaY, double fraction)>();
    // Indice : barriere = b^2/(4a) ; lambda_y = 2d.
    // TODO etudiant : remplir la boucle sur (b, d) avec b croissant et d decroissant.
    // results.Add((..., ..., ...));
    return results;
}
// var res = RecoveryVsConflictingBarrierCurvature();
// foreach (var (barrier, ly, frac) in res)
//     Console.WriteLine(FormattableString.Invariant($"barrier={barrier:F2}  lambda_y={ly:F2}  frac={frac:F3}"));

## 4. Le verdict decisif : correlation partielle a 3 covariables (FWL)

En 2D, la courbure locale n'est pas un scalaire mais un couple de valeurs propres.
Deux resumes scalaires sont candidats :

- $\sigma_{\text{mean}}$ (moyenne des valeurs propres) -- **degenere** : il
  partage $b$ avec la largeur longitudinale, donc structurellement couple a $w$.
- $\sigma_{\text{min}}$ (valeur propre minimale, la direction molle, celle de
  l'escape) -- **decouple** de $w$ par construction (l'axe $y$ est orthogonal).
  C'est le scalaire le plus charitable a l'hypothese *« $\sigma$ cause la
  recuperabilite »*.

Le test decisif isole le pouvoir predictif propre de chaque scalaire apres
controle de **3 covariables** (largeur, barriere, anisotropie) via la regression
des residus (Frisch-Waugh-Lovell). Le verdict est **CONFIRMED-NEGATIVE** si les
deux partialles sont $\approx 0$ et sous le null $p_{95}$.

In [6]:
// Verdict principal, multi-seed (la recuperation stochastique est aleatoire :
// le seed traverse les tirages de Langevin et la permutation du null).
// Profilage numérique : mêmes différences centrales, initialisations et seuils.
System.Collections.Generic.List<(double x, double y, double lo, double hi, double width, double barrier)> Profile2D(double aa, double bb, double dd)
{
    Func<double, double, double> v = (x, y) => aa * Math.Pow(x, 4) - bb * x * x + dd * y * y;
    var pts = new System.Collections.Generic.List<(double x, double y, double lo, double hi)>();
    for (int i = 0; i < 24; i++)
        for (int j = 0; j < 24; j++)
        {
            double x = -2.5 + 5.0 * i / 23, y = -2.5 + 5.0 * j / 23;
            bool ok = false;
            for (int k = 0; k < 50; k++)
            {
                var g = gradient(v, x, y);
                if (Math.Sqrt(g.gx*g.gx + g.gy*g.gy) < 1e-9) { ok = true; break; }
                var h = hessian(v, x, y);
                double det = h.xx*h.yy - h.xy*h.xy;
                if (Math.Abs(det) < 1e-30) break;
                double dx = (h.yy*g.gx - h.xy*g.gy)/det, dy = (h.xx*g.gy - h.xy*g.gx)/det;
                x -= dx; y -= dy;
                if (double.IsNaN(x) || double.IsInfinity(x) || double.IsNaN(y) || double.IsInfinity(y)) break;
            }
            if (!ok || pts.Any(q => Math.Sqrt((q.x-x)*(q.x-x)+(q.y-y)*(q.y-y)) < 1e-3)) continue;
            var hh = hessian(v, x, y);
            double disc = Math.Sqrt((hh.xx-hh.yy)*(hh.xx-hh.yy)+4*hh.xy*hh.xy);
            pts.Add((x, y, (hh.xx+hh.yy-disc)/2, (hh.xx+hh.yy+disc)/2));
        }
    var result = new System.Collections.Generic.List<(double x, double y, double lo, double hi, double width, double barrier)>();
    var saddles = pts.Where(q => q.lo <= 0 && q.hi >= 0).ToArray();
    foreach (var m in pts.Where(q => q.lo > 0))
    {
        if (saddles.Length == 0) continue;
        var s = saddles.OrderBy(q => (q.x-m.x)*(q.x-m.x)+(q.y-m.y)*(q.y-m.y)).First();
        result.Add((m.x, m.y, m.lo, m.hi, Math.Sqrt((m.x-s.x)*(m.x-s.x)+(m.y-s.y)*(m.y-s.y)), v(s.x,s.y)-v(m.x,m.y)));
    }
    return result;
}
// Rangs moyens pour les ex-aequo exacts et corrélation de Pearson.
double[] RankLandscape(double[] values)
{
    int n = values.Length;
    int[] order = Enumerable.Range(0,n).OrderBy(i => values[i]).ToArray();
    var ranks = new double[n];
    for (int i = 0; i < n;)
    {
        int j = i;
        while (j + 1 < n && values[order[j+1]] == values[order[i]]) j++;
        double rank = 0.5 * (i + j) + 1;
        for (int k = i; k <= j; k++) ranks[order[k]] = rank;
        i = j + 1;
    }
    return ranks;
}
double PearsonLandscape(double[] x, double[] y)
{
    if (x.Length < 2) return 0;
    double mx = x.Average(), my = y.Average(), xx = 0, yy = 0, xy = 0;
    for (int i = 0; i < x.Length; i++) { double u=x[i]-mx, v=y[i]-my; xx+=u*u; yy+=v*v; xy+=u*v; }
    return Math.Sqrt(xx/x.Length) < 1e-12 || Math.Sqrt(yy/y.Length) < 1e-12 ? 0 : xy/Math.Sqrt(xx*yy);
}
// Projection orthogonale avec rejet des colonnes dépendantes.
double[] ResidualLandscape(double[] y, double[][] covariates)
{
    int n = y.Length;
    var basis = new System.Collections.Generic.List<double[]>();
    var columns = new System.Collections.Generic.List<double[]> { Enumerable.Repeat(1.0, n).ToArray() };
    columns.AddRange(covariates);
    foreach (var column in columns)
    {
        var q = (double[])column.Clone();
        for (int pass = 0; pass < 2; pass++)
            foreach (var u in basis)
            {
                double dot = 0; for (int i = 0; i < n; i++) dot += q[i]*u[i];
                for (int i = 0; i < n; i++) q[i] -= dot*u[i];
            }
        double norm = Math.Sqrt(q.Sum(z => z*z));
        if (norm <= 1e-10) continue;
        for (int i = 0; i < n; i++) q[i] /= norm;
        basis.Add(q);
    }
    var residual = (double[])y.Clone();
    foreach (var u in basis)
    {
        double dot = 0; for (int i = 0; i < n; i++) dot += residual[i]*u[i];
        for (int i = 0; i < n; i++) residual[i] -= dot*u[i];
    }
    return residual;
}
double PartialLandscape(double[] x, double[] y, params double[][] covariates)
{
    if (x.Length < 2) return 0;
    var rx = RankLandscape(x); var ry = RankLandscape(y);
    if (covariates.Length == 0) return PearsonLandscape(rx, ry);
    var rc = covariates.Select(c => RankLandscape(c)).ToArray();
    return PearsonLandscape(ResidualLandscape(rx, rc), ResidualLandscape(ry, rc));
}
double Percentile95(double[] values)
{
    var sorted = values.OrderBy(x => x).ToArray();
    double pos = 0.95 * (sorted.Length - 1); int lo = (int)pos, hi = (int)Math.Ceiling(pos);
    return sorted[lo] + (pos-lo)*(sorted[hi]-sorted[lo]);
}
double[] ShuffleLandscape(double[] values, LandscapeRandom rng)
{
    var copy = (double[])values.Clone();
    for (int i = copy.Length-1; i > 0; i--)
    {
        int j = (int)(rng.Uniform() * (i+1));
        double t = copy[i]; copy[i] = copy[j]; copy[j] = t;
    }
    return copy;
}
// La géométrie est indépendante de la graine ; ses profils sont calculés une fois.
var landscapeProfiles = new System.Collections.Generic.List<(double a, double b, double d, double x, double mean, double min, double width, double barrier, double aniso)>();
foreach (double aa in new[] {0.5,0.8,1.2,1.8,2.5})
    foreach (double bb in new[] {0.8,1.4,2.0,2.6,3.2})
        foreach (double dd in new[] {0.5,1.5,3.0,4.5,6.0})
        {
            var profiles = Profile2D(aa,bb,dd);
            if (profiles.Count != 2) continue;
            foreach (var q in profiles)
                landscapeProfiles.Add((aa,bb,dd,q.x,(q.lo+q.hi)/2,q.lo,q.width,q.barrier,q.hi/q.lo));
        }
var sigMean = landscapeProfiles.Select(q => q.mean).ToArray();
var sigMin = landscapeProfiles.Select(q => q.min).ToArray();
var widFamily = landscapeProfiles.Select(q => q.width).ToArray();
var barFamily = landscapeProfiles.Select(q => q.barrier).ToArray();
var anisFamily = landscapeProfiles.Select(q => q.aniso).ToArray();
double rhoMinW = PearsonLandscape(RankLandscape(sigMin), RankLandscape(widFamily));
double rhoMeanW = PearsonLandscape(RankLandscape(sigMean), RankLandscape(widFamily));
Console.WriteLine("=== landscape_verdict (2D anisotrope) ===");
foreach (int seed in new[] {0,1,7,42})
{
    // Le générateur de la famille et celui du null sont initialisés séparément, comme dans le module source.
    var familyRng = new LandscapeRandom((ulong)seed);
    var recovery = landscapeProfiles.Select(q => Recover2D(q.x,q.a,q.b,q.d,familyRng)).ToArray();
    double pmin = PartialLandscape(sigMin,recovery,widFamily,barFamily,anisFamily);
    double pmean = PartialLandscape(sigMean,recovery,widFamily,barFamily,anisFamily);
    var nullRng = new LandscapeRandom((ulong)seed);
    var nullValues = new double[100];
    for (int k = 0; k < nullValues.Length; k++)
        nullValues[k] = Math.Abs(PartialLandscape(ShuffleLandscape(sigMean,nullRng),recovery,widFamily,barFamily,anisFamily));
    double p95 = Percentile95(nullValues);
    bool decoupled = Math.Abs(rhoMinW) < 0.2;
    double decisive = Math.Abs(pmean) >= Math.Abs(pmin) ? pmean : pmin;
    string verdict = !decoupled ? "INCONCLUSIVE" : Math.Abs(decisive) > p95 && Math.Abs(decisive) > 0.2 ? "SUBSTRATE-ARTIFACT" : decisive < -0.2 ? "INCONCLUSIVE" : "CONFIRMED-NEGATIVE";
    Console.WriteLine(FormattableString.Invariant($"seed {seed,2}: {verdict,-18} decoupl(sigma_min)={decoupled} (rho_smin_w={rhoMinW:+0.000;-0.000;+0.000}, rho_smean_w={rhoMeanW:+0.000;-0.000;+0.000})"));
    Console.WriteLine(FormattableString.Invariant($"          partial_min={pmin:+0.000;-0.000;+0.000}  partial_mean={pmean:+0.000;-0.000;+0.000}  [null p95={p95:F3}]"));
}

=== landscape_verdict (2D anisotrope) ===


seed  0: CONFIRMED-NEGATIVE decoupl(sigma_min)=True (rho_smin_w=+0.163, rho_smean_w=+0.379)


          partial_min=-0.050  partial_mean=-0.032  [null p95=0.128]


seed  1: CONFIRMED-NEGATIVE decoupl(sigma_min)=True (rho_smin_w=+0.163, rho_smean_w=+0.379)


          partial_min=+0.053  partial_mean=+0.076  [null p95=0.131]


seed  7: CONFIRMED-NEGATIVE decoupl(sigma_min)=True (rho_smin_w=+0.163, rho_smean_w=+0.379)


          partial_min=+0.046  partial_mean=+0.054  [null p95=0.119]


seed 42: CONFIRMED-NEGATIVE decoupl(sigma_min)=True (rho_smin_w=+0.163, rho_smean_w=+0.379)


          partial_min=+0.001  partial_mean=-0.001  [null p95=0.124]


## Lecture 4 -- Quatre graines, un verdict : CONFIRMED-NEGATIVE x4

Pour les quatre graines, le test affiche CONFIRMED-NEGATIVE. La corrélation entre $\sigma_{\min}$ et la largeur vaut **+0.163**, contre +0.379 pour $\sigma_{\text{mean}}$ : le seuil de découplage $|\rho|<0.2$ est atteint pour le premier résumé. Les corrélations de la source Python diffèrent légèrement ; le profilage numérique et le traitement des rangs peuvent influer sur ces mesures.

Dans l'ordre des graines 0, 1, 7 et 42, les partielles de $\sigma_{\min}$ valent -0.050, +0.053, +0.046 et +0.001 ; celles de $\sigma_{\text{mean}}$ valent -0.032, +0.076, +0.054 et -0.001. Les seuils nuls correspondants sont 0.128, 0.131, 0.119 et 0.124 : chaque partielle reste sous son seuil en valeur absolue. Les valeurs stochastiques diffèrent de la source, notamment parce que le générateur utilisé ici n'est pas celui de NumPy.

Sur cette famille et pour ce protocole, aucun des deux résumés ne montre de pouvoir prédictif propre après contrôle de la largeur, de la barrière et de l'anisotropie.

> **Exercice 2** (anisotropie comme 3e covariable). La 3e covariable du controle
> est l'anisotropie $\lambda_{\max}/\lambda_{\min}$. Retrancher cette covariable
> (controler uniquement largeur + barriere, comme en 1D) change-t-il le verdict ?
> Construire un verdict a 2 covariables et comparer. *Indice : en 2D, l'anisotropie
> capture une partie de la structure que $\sigma_{\min}$ seul laisse echapper.*

In [7]:
// Exercice 2 -- a completer.
double? Verdict2Cov(int seed = 0, int nShuffle = 100)
{
    // Indice : recuperer la famille via bl._gather_landscape_family(...), puis
    // appeler partial_spearman avec 2 covariables au lieu de 3.
    // TODO etudiant.
    return null;
}
// var p2 = Verdict2Cov();
// Console.WriteLine(FormattableString.Invariant($"Partielle 2-cov sigma_min : {p2:F3} (vs 3-cov sous le null)."));

## 5. Controle nul : la sous-famille re-couplee reproduit la fronce

Discipline du null model (#9531) : il faut verifier que le protocole detecte le
couplage quand il est la. Si l'on force le **couplage canonique** (a/d fixes,
$b$ varie densement), alors $\sigma \propto w$ et la partielle de $\sigma$ apres
controle de la largeur seule doit retomber a $\approx 0$ (motif fronce) -- signe
que c'etait bien la largeur qui portait le signal.

In [8]:
// Contrôle recouplé : douze valeurs de b, deux minima et permutations de sigma.
var nullSig = new System.Collections.Generic.List<double>();
var nullWid = new System.Collections.Generic.List<double>();
var nullRec = new System.Collections.Generic.List<double>();
var recoupledRng = new LandscapeRandom(0);
for (int i = 0; i < 12; i++)
{
    double bb = 0.6 + (3.6 - 0.6) * i / 11.0;
    var profiles = Profile2D(1.0,bb,2.0);
    if (profiles.Count != 2) continue;
    foreach (var q in profiles)
    {
        nullSig.Add((q.lo+q.hi)/2);
        nullWid.Add(q.width);
        nullRec.Add(Recover2D(q.x,1.0,bb,2.0,recoupledRng));
    }
}
var ns = nullSig.ToArray(); var nw = nullWid.ToArray(); var nr = nullRec.ToArray();
double rhoNull = PearsonLandscape(RankLandscape(ns),RankLandscape(nw));
double partialNull = PartialLandscape(ns,nr,nw);
var permutedNull = new double[100];
for (int k = 0; k < permutedNull.Length; k++)
    permutedNull[k] = Math.Abs(PartialLandscape(ShuffleLandscape(ns,recoupledRng),nr,nw));
double thresholdNull = Percentile95(permutedNull);
bool fronce = rhoNull > 0.6 && Math.Abs(partialNull) < thresholdNull;
Console.WriteLine("Controle nul re-couple (a/d fixes, b dense) :");
Console.WriteLine(FormattableString.Invariant($"  rho(sigma, width)        = {rhoNull:+0.000;-0.000;+0.000}   (re-couple : > 0.6 attendu)"));
Console.WriteLine(FormattableString.Invariant($"  partial(sigma | width)   = {partialNull:+0.000;-0.000;+0.000}   (fronce : |.| < null_p95={thresholdNull:F3} attendu)"));
Console.WriteLine($"  reproduit motif fronce  = {fronce}");

Controle nul re-couple (a/d fixes, b dense) :


  rho(sigma, width)        = +0.995   (re-couple : > 0.6 attendu)


  partial(sigma | width)   = -0.086   (fronce : |.| < null_p95=0.412 attendu)


  reproduit motif fronce  = True


## Lecture 5 -- Le contrôle nul : rho +0.995, motif fronce reproduit

Lorsque l'on force le couplage canonique, la corrélation entre $\sigma$ et la largeur vaut **+0.995**, au-dessus du seuil de 0.6. La partielle après contrôle de la largeur vaut -0.086, sous le seuil nul de 0.412 en valeur absolue : le test affiche `True` pour le motif fronce.

La corrélation, la partielle et le seuil diffèrent de ceux de la source Python. Les résultats stochastiques diffèrent notamment parce que les générateurs ne sont pas les mêmes. Ce contrôle illustre le comportement du protocole dans une famille recouplée ; il ne suffit pas à établir à lui seul que largeur et barrière déterminent toute la récupérabilité.

> **Exercice 3** (le null a 3 covariables aussi). Le null ci-dessus controle la
> largeur seule (1 covariable). Refaire le meme null mais en controlant largeur +
> barriere + anisotropie (3 covariables), comme le verdict principal. La partielle
> de $\sigma$ reste-t-elle sous le null ? *Indice : dans le null re-couple, a/d
> fixes, l'anisotropie est elle-meme fixe -- quelle consequence sur le rang de la
> matrice de design ?*

In [9]:
// Exercice 3 -- a completer.
double? Null3Cov(int seed = 0, int nShuffle = 100)
{
    // Indice : dans le null, a/d fixes -> anisotropie = lambda_max/lambda_min fixe.
    // TODO etudiant : que se passe-t-il quand une covariable est constante ?
    return null;
}
// var p3 = Null3Cov();
// Console.WriteLine(FormattableString.Invariant($"Partielle 3-cov sigma_min (null) : {p3:F3}"));

## Vérifiez votre compréhension

1. Le knob transverse `d` fait varier le régime anisotrope en laissant la largeur inchangée au bruit numérique près. Pourquoi le découplage atteint (`|rho| < 0.2`) donne-t-il au régime 2D l'occasion la plus favorable à l'hypothèse σ ?
2. Avec le scalaire le plus charitable, la graine 42 rend `partial_min = +0.001` pour un null p95 de `0.124`, et le contrôle recouplé donne `rho = +0.995` avec une partielle `-0.086` sous le null p95 `0.412`. Expliquez pourquoi le premier est un verdict et le second un contrôle.
3. La mesure non triviale rend `std = 0.187` sur les récupérations stochastiques. Qu'est-ce qui distingue cette dispersion de celle étudiée dans ICT-15h, et pourquoi la comparabilité exige-t-elle le même protocole de Langevin ?

## À retenir

Les trois régimes étudiés -- symétrique 1D (15f), asymétrique 1D (15h) et anisotrope 2D (15i) -- convergent vers le meme verdict :

> **Sur une famille de substrats ou la courbure locale $\sigma$ varie
> independamment de la largeur du puits, aucun resume scalaire de $\sigma$ (en 1D
> le scalaire unique, en 2D $\sigma_{\min}$ ou $\sigma_{\text{mean}}$) n'a de
> pouvoir predictif propre pour la recuperabilite, apres controle de la largeur, de
> la barriere (et de l'anisotropie en 2D).**

Le regime 2D etait le plus favorable a l'hypothese : il offrait un knob transverse
orthogonal ($d$) permettant de decoupler $\sigma_{\min}$ de la largeur -- un
decouplage par construction **atteint** ici ($|\rho|<0.2$), que le regime
asymetrique 1D ne pouvait pas realiser. Et malgre cela, avec le scalaire le plus
charitable, le verdict reste negatif. La recuperabilite est determinee par la
**largeur du puits et la barriere** (geometrie globale), non par la courbure locale
(qu'elle soit scalaire, minimale, ou moyenne). Le Pont #1-bis est clos.

## Pour aller plus loin

- Les chantiers 1/3 et 2/3 du Pont #1-bis : [ICT-15f](ICT-15f-Bridge1bis-DecoupledFamily-Python.ipynb) (symétrique) et [ICT-15h](ICT-15h-Bridge1bis-AsymmetricFamily-Python.ipynb) (asymétrique).
- **Jumeau Python** → [ICT-15i-Bridge1bis-2DLandscape-Python](ICT-15i-Bridge1bis-2DLandscape-Python.ipynb) — le carnet d'origine, en Python.
- Revenir au [cadrage de la série](../ICT-0-Framing.md).

## Attestation de parité Python / C#

Les écarts ci-dessous sont relevés sur les sorties affichées, à leur précision d'affichage ; ils désignent **C# moins Python**. Les tirages stochastiques utilisent les mêmes graines déclarées mais un générateur différent de celui de NumPy : leurs écarts ne constituent pas une égalité numérique des trajectoires.

- **Géométrie canonique** : écart nul aux trois décimales affichées pour les deux courbures, l'anisotropie, la largeur et la barrière.
- **Famille transverse** : écart nul aux précisions du tableau pour les quatre largeurs, valeurs propres minimales et anisotropies. L'écart maximal de largeur vaut 4.68e-13 en C# contre 4.32e-13 en Python, soit une différence de 0.36e-13, au niveau du bruit numérique.
- **Récupération stochastique** : écart d'écart-type −0.016 ; sans kick +0.066, avec kick −0.083 ; les deux contrôles physiques donnent respectivement −0.075 et +0.037 à partir des valeurs affichées. Le booléen du contrôle est `False` dans les deux carnets.
- **Verdict multi-graines** : les quatre verdicts et le découplage concordent. Les corrélations avec la largeur diffèrent de +0.015 pour $\sigma_{\min}$ et de +0.006 pour $\sigma_{\text{mean}}$. Pour les graines 0, 1, 7 et 42, les écarts des partielles minimales sont respectivement +0.021, +0.015, +0.114 et +0.135 ; ceux des partielles moyennes sont +0.063, +0.041, +0.178 et +0.130 ; ceux des seuils nuls sont +0.010, +0.002, +0.014 et +0.025.
- **Contrôle recouplé** : écarts de corrélation +0.002, de partielle −0.400 et de seuil nul −0.024 ; le motif est déclaré reproduit dans les deux carnets.
- **Exercices** : les trois cellules sont des stubs sans résultat calculé.